# Performing DEG Pathway analysis (limma, R) on Pseudobulking from atlas
* Ovarian cancer samples only
* dataframes made previously (2026-04-10_PseudobulkingSampleType_fromNRcode.jpynb), separate for each DC subset and each has seperate metadata file
* Following code from NR 2025-02-12_Pseudo_Cancer_DEGS.ipynb

In [1]:
library(limma)
library(edgeR)
library(dplyr)
library(RColorBrewer)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
# set directory
setwd('/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/Pseudobulk')

## Load data - starting with the mregDC

In [3]:
list.files()

[1] "dc1_limma_DEG.csv"                           
 [2] "dc2_limma_DEG.csv"                           
 [3] "mregdc_limma_DEG.csv"                        
 [4] "OC_mregDC"                                   
 [5] "oc_pseudobulk_DC1_data.csv"                  
 [6] "oc_pseudobulk_DC2_data.csv"                  
 [7] "oc_pseudobulk_mregDC_data.csv"               
 [8] "oc_pseudobulk_sampletype_metadata_DC1.csv"   
 [9] "oc_pseudobulk_sampletype_metadata_DC2.csv"   
[10] "oc_pseudobulk_sampletype_metadata_mregDC.csv"
[11] "pseudobulk_DC1_data.csv"                     
[12] "pseudobulk_DC2_data.csv"                     
[13] "pseudobulk_metadata_DC1.csv"                 
[14] "pseudobulk_metadata_DC2.csv"                 
[15] "pseudobulk_metadata_mregDC.csv"              
[16] "pseudobulk_mregDC_data.csv"                  
[17] "pseudobulk_sampletype_metadata_DC1.csv"      
[18] "pseudobulk_sampletype_metadata_DC2.csv"      
[19] "pseudobulk_sampletype_metadata_mregDC.csv"   
[20] "PT-MT_limma.csv"

In [4]:
#load data
sampleinfo <- read.csv("oc_pseudobulk_sampletype_metadata_mregDC.csv", row.names =1)
head(sampleinfo)

,sample_type_major2
,<chr>
GSE180661_SPECTRUM-OV-002_S1_CD45P_INFRACOLIC_OMENTUM,metastatic tumour
GSE180661_SPECTRUM-OV-002_S1_CD45P_RIGHT_OVARY,primary tumour
GSE180661_SPECTRUM-OV-003_S1_UNSORTED_INFRACOLIC_OMENTUM,metastatic tumour
GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_ADNEXA,primary tumour
GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_UPPER_QUADRANT,metastatic tumour
GSE180661_SPECTRUM-OV-003_S1_UNSORTED_PELVIC_PERITONEUM,metastatic tumour


In [5]:
dim(sampleinfo)

[1] 167   1

In [6]:
cts_mregDC <- read.csv("oc_pseudobulk_mregDC_data.csv")
head(cts_mregDC)

,X,GSE180661_SPECTRUM.OV.002_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM.OV.002_S1_CD45P_RIGHT_OVARY,GSE180661_SPECTRUM.OV.003_S1_UNSORTED_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM.OV.003_S1_UNSORTED_LEFT_ADNEXA,GSE180661_SPECTRUM.OV.003_S1_UNSORTED_LEFT_UPPER_QUADRANT,GSE180661_SPECTRUM.OV.003_S1_UNSORTED_PELVIC_PERITONEUM,GSE180661_SPECTRUM.OV.003_S1_UNSORTED_RIGHT_ADNEXA,GSE180661_SPECTRUM.OV.003_S1_UNSORTED_RIGHT_UPPER_QUADRANT,GSE180661_SPECTRUM.OV.008_S1_CD45P_BOWEL,⋯,PRJCA005422_HGSOC6_AS,PRJCA005422_HGSOC6_MT,PRJCA005422_HGSOC6_PT,PRJCA005422_HGSOC7_PT,PRJCA005422_HGSOC8_AS,PRJCA005422_HGSOC8_PT,PRJCA005422_HGSOC9_AS,PRJCA005422_HGSOC9_PT,PRJCA005422_HGSOC10_AS,PRJCA005422_HGSOC10_PT
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,FAM87B,0,0,0,0,0,0,0,0,0,⋯,0,1,0,0,0,0,0,0,0,0
2,LINC00115,0,0,0,0,0,0,0,1,0,⋯,1,1,0,1,0,0,1,0,0,0
3,FAM41C,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
4,SAMD11,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
5,NOC2L,0,0,1,0,0,1,0,1,2,⋯,0,2,0,1,0,0,3,0,0,0
6,KLHL17,0,0,1,0,0,1,0,0,0,⋯,0,1,0,0,0,0,1,0,1,0


In [7]:
dim(cts_mregDC)

[1] 15074   168

In [8]:
# import csv
tt1_limma <- read.csv("PT-MT_limma.csv")

In [9]:
head(tt1_limma)

,X,Symbol,logFC,AveExpr,t,P.Value,adj.P.Val,B
,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,TYROBP,13983,5.569684,10.029229,4.889259,2.067751e-06,0.01768548,4.389553
2,RPS15A,11428,4.338221,10.945403,4.648081,6.057856e-06,0.02590642,3.476500
3,SLC25A6,6190,4.443334,8.739950,4.419996,1.613761e-05,0.03369127,2.630757
4,RPS3A,3890,4.188353,11.250495,4.293620,2.733791e-05,0.03369127,2.201090
5,ZC3H12C,8499,-4.432971,3.956478,-4.274173,2.961797e-05,0.03369127,2.127371
6,RPS8,477,3.588908,11.325386,4.221489,3.674576e-05,0.03369127,1.969344


## Read in all contrasts generated previously 
(2026-04-24-OC_Pseduo_pathways_PrimaryvMetsv2.ipynb)

In [10]:
# import csvs for differentially up and down genes
df_up_genes <- read.csv("OC_mregDC/df_up_genes.csv")
df_down_genes <- read.csv("OC_mregDC/df_down_genes.csv")

In [11]:
tt1_mregDC <- read.csv("OC_mregDC/tt1_mregDC.csv")
tt2_mregDC <- read.csv("OC_mregDC/tt2_mregDC.csv")
tt3_mregDC <- read.csv("OC_mregDC/tt3_mregDC.csv")

In [12]:
class(tt1_mregDC)

[1] "data.frame"

In [13]:
head(tt1_mregDC)

,X.1,X,Symbol,logFC,AveExpr,t,P.Value,adj.P.Val,B
,<int>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,13983,TYROBP,13983,5.569684,10.029229,4.889259,2.067751e-06,0.01768548,4.389553
2,11428,RPS15A,11428,4.338221,10.945403,4.648081,6.057856e-06,0.02590642,3.476500
3,6190,SLC25A6,6190,4.443334,8.739950,4.419996,1.613761e-05,0.03369127,2.630757
4,3890,RPS3A,3890,4.188353,11.250495,4.293620,2.733791e-05,0.03369127,2.201090
5,8499,ZC3H12C,8499,-4.432971,3.956478,-4.274173,2.961797e-05,0.03369127,2.127371
6,477,RPS8,477,3.588908,11.325386,4.221489,3.674576e-05,0.03369127,1.969344


## Performing GSEA using clusterprofiler

In [14]:
library(clusterProfiler)
library(enrichplot)
library(dplyr)
library(msigdbr)
library('org.Hs.eg.db')



clusterProfiler v4.14.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

S Xu, E Hu, Y Cai, Z Xie, X Luo, L Zhan, W Tang, Q Wang, B Liu, R Wang,
W Xie, T Wu, L Xie, G Yu. Using clusterProfiler to characterize
multiomics data. Nature Protocols. 2024, 19(11):3292-3320


Attaching package: ‘clusterProfiler’


The following object is masked from ‘package:stats’:

    filter


enrichplot v1.26.1 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

Guangchuang Yu. Using meshes for MeSH term enrichment and semantic
analyses. Bioinformatics. 2018, 34(21):3766-3767,
doi:10.1093/bioinformatics/bty410

Loading required package: AnnotationDbi

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following object is masked from ‘package:limma’:

    plotMA


The following objects

## Show which genes sets will be used for the comparison

From the meeting with KR-NR-KT on 13/4/26
* IFN --> HALLMARK
* TNF --> HALLMARK
* Ag presentation --> KEGG
* Migration --> KEGG
* Phagocytosis --> KEGG

Load the packages to look at the KEGG and HALLMARK gene set pathways

In [ ]:
#read in genes lists to be used from stored data
